In [19]:
import pandas as pd
import re
import pickle
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
df = pd.read_csv("spam (1).csv", encoding="cp1252")

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Convert labels
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

In [21]:
ps = PorterStemmer()

def clean_text(text):

    text = re.sub('[^a-zA-Z]', ' ', str(text))

    text = text.lower()

    words = text.split()

    words = [
        ps.stem(word)
        for word in words
        if word not in stopwords.words('english')
    ]

    return " ".join(words)

df['message'] = df['message'].apply(clean_text)


In [22]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['message'])

y = df['label']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [24]:
rf = RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("Accuracy:",accuracy_score(y_test, pred_rf))


Accuracy: 0.979372197309417


In [25]:
pickle.dump(
    rf,
    open("phishing_model.pkl", "wb")
)

In [26]:
pickle.dump(
    tfidf,
    open("vectorizer.pkl", "wb")
)

In [27]:
loaded_model = pickle.load(
    open("phishing_model.pkl", "rb")
)

loaded_vectorizer = pickle.load(
    open("vectorizer.pkl", "rb")
)


In [ ]:
message = ["claim","urgent","free","get","win","prize","send","link","click here"]

message = clean_text(message)

message_vector = loaded_vectorizer.transform([message])

prediction = loaded_model.predict(message_vector)

probability = loaded_model.predict_proba(message_vector)

spam_probability = probability[0][1] * 100


In [29]:
if prediction[0] == 1:
    print("Spam / Phishing Message")
else:
    print("Legitimate Message")

print(f"Spam Probability: {spam_probability:.2f}%")

if spam_probability >= 90:
    print("Risk Level: Very High")
elif spam_probability >= 70:
    print("Risk Level: High")
elif spam_probability >= 40:
    print("Risk Level: Medium")
else:
    print("Risk Level: Low")

Spam / Phishing Message
Spam Probability: 76.00%
Risk Level: High


In [30]:
import pickle

# Load model and vectorizer
model = pickle.load(open("phishing_model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

# Test message
message = "Please share your OTP"

# Clean message
message = clean_text(message)

# Convert to TF-IDF
message_vector = vectorizer.transform([message])

# Predict
prediction = model.predict(message_vector)

# Probability
probability = model.predict_proba(message_vector)

spam_probability = probability[0][1] * 100

print("Prediction:", prediction[0])
print(f"Spam Probability: {spam_probability:.2f}%")

Prediction: 0
Spam Probability: 4.00%


In [31]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.979372197309417
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       1.00      0.85      0.92       150

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

